In [1]:
import warnings
warnings.filterwarnings("ignore")
import os
os.environ["NIXTLA_ID_AS_COL"] = "true"
import numpy as np
np.set_printoptions(suppress=True)
np.random.seed(1)
import random
random.seed(1)
import pandas as pd
pd.set_option("max_colwidth", 100)
pd.set_option("display.precision", 3)
from utilsforecast.plotting import plot_series as plot_series_utils
import seaborn as sns
sns.set_style("whitegrid")
import matplotlib.pyplot as plt
plt.style.use("ggplot")
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 100,
    "savefig.dpi": 300,
    "figure.constrained_layout.use": True,
    "axes.titlesize": 12,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 9,
    "legend.title_fontsize": 10,
    "grid.alpha": 1.0,
})
import matplotlib as mpl
from cycler import cycler
mpl.rcParams['axes.prop_cycle'] = cycler(color=["#000000", "#000000"])
from fpppy.utils import plot_series

mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#2f2fff"], name="black_and_blue"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00"], name="black_and_orange"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#000000"], name="black"),
    force=True,
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#0072B2", "#D55E00"],
        name='black_and_2color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00", "#0072B2", "#009E73"],
        name='black_and_3color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#000000", "#D55E00", "#0072B2", "#009E73", "#CC79A7"],
        name='black_and_4color',
    ),
    force=True
)
mpl.colormaps.register(
    mpl.colors.ListedColormap(
        ["#D55E00", "#0072B2", "#009E73", "#CC79A7"],
        name='r_colors',
    ),
    force=True
)

In [2]:
import statsmodels.api as sm
from scipy.stats import pearsonr
from statsmodels.graphics.tsaplots import plot_acf

In [3]:
df = pd.DataFrame({
    "Year": list(range(2015, 2020)),
    "Observation": [123, 39, 78, 52, 110]
})

In [6]:
print(df.index)
print(df.columns)

RangeIndex(start=0, stop=5, step=1)
Index(['Year', 'Observation'], dtype='str')


In [7]:
df.dtypes

Year           int64
Observation    int64
dtype: object

In [8]:
print(type(df['Year']))
print(df['Year'])

<class 'pandas.Series'>
0    2015
1    2016
2    2017
3    2018
4    2019
Name: Year, dtype: int64


In [9]:
year_df = df.set_index("Year")
year_df

,Observation
Year,
2015,123
2016,39
2017,78
2018,52
2019,110


## Timestamps and Periods

In [10]:
print(repr(pd.Timestamp("2020-01")))
print(repr(pd.Period("2020-01")))

Timestamp('2020-01-01 00:00:00')
Period('2020-01', 'M')


In [15]:
print(repr(pd.Timestamp("2020-01-23")))
print(repr(pd.Timestamp("2020-02")))
print(repr(pd.Timestamp("2023")))

Timestamp('2020-01-23 00:00:00')
Timestamp('2020-02-01 00:00:00')
Timestamp('2023-01-01 00:00:00')


In [16]:
print(repr(pd.Period("2020")))
print(repr(pd.Period("2023-02")))
print(repr(pd.Period("2021-03-01")))

Period('2020', 'Y-DEC')
Period('2023-02', 'M')
Period('2021-03-01', 'D')


### Time Series

#### `pd.to_datetime()` and `pd.date_range()`

In [19]:
ts_few = pd.to_datetime(["2020-01-01", "2020-01-02", "2020-01-03"])
ts_range = pd.date_range(start="2021-02-01", end="2021-03-02", freq="2h")

print(ts_few)
print(ts_range)

DatetimeIndex(['2020-01-01', '2020-01-02', '2020-01-03'], dtype='datetime64[us]', freq=None)
DatetimeIndex(['2021-02-01 00:00:00', '2021-02-01 02:00:00',
               '2021-02-01 04:00:00', '2021-02-01 06:00:00',
               '2021-02-01 08:00:00', '2021-02-01 10:00:00',
               '2021-02-01 12:00:00', '2021-02-01 14:00:00',
               '2021-02-01 16:00:00', '2021-02-01 18:00:00',
               ...
               '2021-03-01 06:00:00', '2021-03-01 08:00:00',
               '2021-03-01 10:00:00', '2021-03-01 12:00:00',
               '2021-03-01 14:00:00', '2021-03-01 16:00:00',
               '2021-03-01 18:00:00', '2021-03-01 20:00:00',
               '2021-03-01 22:00:00', '2021-03-02 00:00:00'],
              dtype='datetime64[us]', length=349, freq='2h')


In [21]:
print(ts_range.to_period())
print(ts_range.to_period(freq='D'))
print(ts_range.to_period(freq='W'))
print(ts_range.to_period().to_timestamp())

PeriodIndex(['2021-02-01 00:00', '2021-02-01 02:00', '2021-02-01 04:00',
             '2021-02-01 06:00', '2021-02-01 08:00', '2021-02-01 10:00',
             '2021-02-01 12:00', '2021-02-01 14:00', '2021-02-01 16:00',
             '2021-02-01 18:00',
             ...
             '2021-03-01 06:00', '2021-03-01 08:00', '2021-03-01 10:00',
             '2021-03-01 12:00', '2021-03-01 14:00', '2021-03-01 16:00',
             '2021-03-01 18:00', '2021-03-01 20:00', '2021-03-01 22:00',
             '2021-03-02 00:00'],
            dtype='period[2h]', length=349)
PeriodIndex(['2021-02-01', '2021-02-01', '2021-02-01', '2021-02-01',
             '2021-02-01', '2021-02-01', '2021-02-01', '2021-02-01',
             '2021-02-01', '2021-02-01',
             ...
             '2021-03-01', '2021-03-01', '2021-03-01', '2021-03-01',
             '2021-03-01', '2021-03-01', '2021-03-01', '2021-03-01',
             '2021-03-01', '2021-03-02'],
            dtype='period[D]', length=349)
PeriodIndex(['2

In [22]:
print(ts_few.strftime("%m/%d/%Y"))
print(ts_range.to_period().strftime("%Y %b ~ %H:%M"))

Index(['01/01/2020', '01/02/2020', '01/03/2020'], dtype='str')
Index(['2021 Feb ~ 00:00', '2021 Feb ~ 02:00', '2021 Feb ~ 04:00',
       '2021 Feb ~ 06:00', '2021 Feb ~ 08:00', '2021 Feb ~ 10:00',
       '2021 Feb ~ 12:00', '2021 Feb ~ 14:00', '2021 Feb ~ 16:00',
       '2021 Feb ~ 18:00',
       ...
       '2021 Mar ~ 06:00', '2021 Mar ~ 08:00', '2021 Mar ~ 10:00',
       '2021 Mar ~ 12:00', '2021 Mar ~ 14:00', '2021 Mar ~ 16:00',
       '2021 Mar ~ 18:00', '2021 Mar ~ 20:00', '2021 Mar ~ 22:00',
       '2021 Mar ~ 00:00'],
      dtype='str', length=349)
